In [1]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

# ========================
# PARTE 1: CONFIGURAÇÕES
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = f"/content/drive/MyDrive/WebScraping/FEITO/CARRETAO/COLETA_CARRETAO/{data_nome}"
pasta_imagens = os.path.join(base_drive, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

# ========================
# PARTE 2: COLETA DE LINKS
# ========================
url_categoria = "https://carretaoconstrucao.com.br/pisos-e-revestimentos.html"
links_produtos = set()

while url_categoria:
    print(f"Coletando página: {url_categoria}")
    response = requests.get(url_categoria)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    for a in soup.select("a.product-item-link"):
        href = a.get("href")
        if href and href.startswith("http"):
            links_produtos.add(href)

    next_link = soup.find("link", rel="next")
    if next_link:
        url_categoria = next_link.get("href")
        time.sleep(1)
    else:
        url_categoria = None

links_produtos = sorted(list(links_produtos))
print(f"\nTotal de links encontrados: {len(links_produtos)}\n")

# ========================
# PARTE 3: EXTRAÇÃO + SALVAMENTO PARCIAL
# ========================
todos = []
for i, link in enumerate(links_produtos, 1):
    try:
        print(f"Coletando dados de: {link}")
        resp = requests.get(link)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Nome
        nome_tag = soup.find("h1", class_="page-title") or soup.find("h1", {"data-ui-id": "page-title-wrapper"})
        nome = nome_tag.get_text(strip=True) if nome_tag else None

        # Código
        codigo = None
        sku_div = soup.find("div", class_="product attribute sku")
        if sku_div:
            value_div = sku_div.find("div", class_="value")
            if value_div:
                codigo = value_div.get_text(strip=True)
        if not codigo:
            codigo = f"produto_{i}"

        # Preço
        preco_tag = soup.find("span", {"data-price-type": "finalPrice"})
        preco_promocional = preco_tag.get("data-price-amount") if preco_tag else None

        # Descrição
        descricao_tag = soup.find("div", class_="product attribute description")
        descricao = descricao_tag.get_text(strip=True) if descricao_tag else None

        # Imagem
        imagem_tag = soup.find("meta", {"property": "og:image"})
        imagem = imagem_tag.get("content") if imagem_tag else None

        # URL
        url_produto = soup.find("meta", {"property": "og:url"})
        url_produto = url_produto.get("content") if url_produto else link

        # Loja
        loja = "Carretão Construção"

        # Baixar imagem
        caminho_img = os.path.join(pasta_imagens, f"{codigo}.png")
        if not os.path.exists(caminho_img) and imagem not in [None, "N/A"]:
            try:
                img = requests.get(imagem, stream=True, timeout=10)
                if img.status_code == 200:
                    with open(caminho_img, 'wb') as f:
                        for chunk in img.iter_content(1024):
                            f.write(chunk)
            except Exception as e:
                print(f"⚠️ Erro ao baixar imagem {codigo}: {e}")

        # Adicionar aos resultados
        dados_produto = {
            "Código": codigo,
            "Nome": nome,
            "Preço Promocional": preco_promocional,
            "Descrição": descricao,
            "Imagem": imagem,
            "URL": url_produto,
            "Loja": loja
        }
        todos.append(dados_produto)

        # Salvar a cada 100 produtos ou no final
        if i % 100 == 0 or i == len(links_produtos):
            df = pd.DataFrame(todos)
            colunas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Imagem', 'URL', 'Loja']
            df = df[colunas]
            df.to_csv(os.path.join(base_drive, f"carretao_parcial_{i}.csv"), index=False, sep=";")
            df.to_excel(os.path.join(base_drive, f"carretao_parcial_{i}.xlsx"), index=False)
            print(f"💾 Dados salvos com {i} produtos.")
            todos.clear()

    except Exception as e:
        print(f"❌ Erro inesperado no produto {i}:", e)

# ========================
# PARTE 4: UNIFICAÇÃO FINAL
# ========================
print("\n📁 Unificando arquivos CSV...")
csvs = [f for f in os.listdir(base_drive) if f.endswith(".csv") and "FINAL" not in f]
df_final = pd.DataFrame()

for csv in csvs:
    df = pd.read_csv(os.path.join(base_drive, csv), sep=";")
    df_final = pd.concat([df_final, df], ignore_index=True)

csv_final = os.path.join(base_drive, f"carretao_FINAL_{data_nome}.csv")
excel_final = os.path.join(base_drive, f"carretao_FINAL_{data_nome}.xlsx")

df_final.to_csv(csv_final, index=False, sep=";")
df_final.to_excel(excel_final, index=False)

print(f"✅ CSV final salvo: {csv_final}")
print(f"✅ Excel final salvo: {excel_final}")


Coletando página: https://carretaoconstrucao.com.br/pisos-e-revestimentos.html
Coletando página: https://carretaoconstrucao.com.br/pisos-e-revestimentos.html?p=2

Total de links encontrados: 17

Coletando dados de: https://carretaoconstrucao.com.br/nicho-marmore-sintetico-30x30-preto-cozimax.html
Coletando dados de: https://carretaoconstrucao.com.br/nicho-marmore-sintetico-30x30cm-bege-89976-cozimax.html
Coletando dados de: https://carretaoconstrucao.com.br/nicho-marmore-sintetico-37x35cm-branco-110030958-irm-os-corso.html
Coletando dados de: https://carretaoconstrucao.com.br/nicho-marmore-sintetico-37x35cm-preto-110030959-irm-os-corso.html
Coletando dados de: https://carretaoconstrucao.com.br/nicho-marmore-sintetico-40x30-bege-cozimax.html
Coletando dados de: https://carretaoconstrucao.com.br/nicho-marmore-sintetico-40x30cm-branco-cozimax.html
Coletando dados de: https://carretaoconstrucao.com.br/nicho-marmore-sintetico-40x30cm-preto-cozimax.html
Coletando dados de: https://carretaoco